In [1]:
import datetime
import logging  # 追加：ログ制御用
import warnings  # 追加：警告非表示用
import pandas as pd
import yfinance as yf
from IPython.display import display, HTML

# 警告メッセージ（FutureWarningなど）を非表示にする
warnings.filterwarnings("ignore")

# yfinanceの内部エラー出力を非表示にする
yf_logger = logging.getLogger('yfinance')
yf_logger.setLevel(logging.CRITICAL)

# Define the lookback periods (in business days) for performance calculation.
LOOKBACK_DAYS = {
    "1 Day (%)": -2,
    "3 Days (%)": -4,
    "1 Week (%)": -6,
    "1 Month (%)": -21,
    "6 Months (%)": -126,
    "1 Year (%)": -252,
}

# =====================================================================
# DATA PROCESSING FUNCTION (Supports Market Segmentation)
# =====================================================================
def show_market_performances(market_dict):
    today = datetime.date.today()
    start_date = today - datetime.timedelta(days=60)
    
    # Format numerical values to 2 decimal places
    format_config = {"Latest Price": "{:,.2f}"}
    for column_name in LOOKBACK_DAYS.keys():
        format_config[column_name] = "{:+.2f}%"
        
    # ハイライト用の条件関数
    def highlight_extreme_values(val):
        if pd.isna(val):
            return ''
        if val >= 10:
            return 'background-color: #ffcccc; color: #cc0000; font-weight: bold;'
        elif val <= -10:
            return 'background-color: #cce5ff; color: #004085; font-weight: bold;'
        return ''
        
    # Process each market section
    for market_name, ticker_dict in market_dict.items():
        records = []
        
        # Display market header
        display(HTML(f"<h3 style='margin-top: 20px; color: #333;'>■ {market_name} Market</h3>"))
        
        for name, ticker in ticker_dict.items():
            try:
                # 【修正】 ignore_tz=True を追加してタイムゾーン警告を抑制
                df = yf.download(ticker, start=start_date, end=today, progress=False, ignore_tz=True)
                
                if df.empty:
                    # 元のコードにあったWarning表示も邪魔ならコメントアウトしてください
                    # print(f"[Warning] No data found for: {name} ({ticker})")
                    continue
                    
                prices = df["Close"].squeeze()
                latest_price = prices.iloc[-1]
                
                row = {
                    "Name": name,
                    "Ticker": ticker,
                    "Latest Price": latest_price
                }
                
                for label, index in LOOKBACK_DAYS.items():
                    if len(prices) >= abs(index):
                        past_price = prices.iloc[index]
                    else:
                        past_price = prices.iloc[0]
                    
                    row[label] = ((latest_price - past_price) / past_price) * 100
                    
                records.append(row)
                
            except Exception as e:
                # エラー時のprintも非表示にしたい場合はコメントアウトしてください
                pass
        
        if records:
            df_performance = pd.DataFrame(records)
            
            # .map を使用
            styled_df = (df_performance.style
                         .format(format_config)
                         .map(highlight_extreme_values, subset=list(LOOKBACK_DAYS.keys()))
                         .hide(axis="index"))
            
            display(styled_df)
        else:
            print("No data available for this market.")

In [2]:
# =====================================================================
# CONFIGURATION & EXECUTION (Segmented by Market)
# =====================================================================
MARKET_TICKERS = {
    "United States": {
        "S&P 500": "^GSPC",
        "NASDAQ Composite": "^IXIC",
        "Apple": "AAPL",
        "NVIDIA": "NVDA",
        "Microsoft": "MSFT",
    },
    "Japan": {
        "Nikkei 225": "^N225",
        "Toyota Motor": "7203.T",
        "Sony Group": "6758.T",
        "Mitsubishi UFJ Financial": "8306.T",
    },
    "Singapore": {
        "STI Index": "^STI",
        "DBS Group": "D05.SI",
        "UOB": "U11.SI",
        "Singtel": "Z74.SI",
        "CapitaLand Ascendas REIT": "A17U.SI",
    },
    "Japan Stock": {
        "小松製作所 (6301)": "6301.T",
        "ニデック (6594)": "6594.T",
        "未来工業 (7931)": "7931.T",
        "東部ネットワーク (9036)": "9036.T",
        "ニトリHD (9843)": "9843.T",
        "MRK HD (9980)": "9980.T",
    },
    "US Stock (ETF)": {
        "iシェアーズ コア米国総合債券ETF (AGG)": "AGG"
    }
}

# Run and display the segmented tables
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
S&P 500,^GSPC,"7,354.02",-0.05%,-0.16%,-1.95%,-2.77%,+2.01%,+2.01%
NASDAQ Composite,^IXIC,"25,297.62",-0.24%,-1.13%,-4.60%,-6.02%,+1.63%,+1.63%
Apple,AAPL,283.78,+3.14%,-3.57%,-4.78%,-9.19%,+4.68%,+4.68%
NVIDIA,NVDA,192.53,-1.64%,-3.75%,-8.62%,-10.03%,-3.42%,-3.42%
Microsoft,MSFT,372.97,+5.71%,-0.26%,-1.69%,-12.65%,-8.34%,-8.34%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Nikkei 225,^N225,"69,360.88",-4.15%,-0.61%,-2.65%,+4.57%,+17.00%,+17.00%
Toyota Motor,7203.T,"2,768.00",+2.50%,+2.29%,-0.31%,-9.01%,-8.44%,-8.44%
Sony Group,6758.T,"3,199.00",+0.06%,+1.27%,+1.88%,-7.11%,+2.76%,+2.76%
Mitsubishi UFJ Financial,8306.T,"3,245.00",+0.37%,-0.67%,-1.01%,+8.20%,+15.19%,+15.19%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
STI Index,^STI,"5,191.73",-0.52%,-0.27%,-0.02%,+4.06%,+5.68%,+5.68%
DBS Group,D05.SI,65.43,-0.97%,-1.30%,-0.80%,+5.65%,+13.41%,+13.41%
UOB,U11.SI,39.80,-0.28%,-0.10%,+1.40%,+5.60%,+10.10%,+10.10%
Singtel,Z74.SI,4.43,+0.68%,+1.84%,+1.84%,+1.61%,-3.49%,-3.49%
CapitaLand Ascendas REIT,A17U.SI,2.54,+0.40%,+0.79%,+1.20%,+1.20%,+2.01%,+2.01%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
小松製作所 (6301),6301.T,"6,329.00",+1.26%,-3.21%,-2.51%,-3.77%,-3.48%,-3.48%
ニデック (6594),6594.T,"2,543.00",-4.18%,-8.82%,-5.64%,-8.92%,+6.09%,+6.09%
未来工業 (7931),7931.T,"3,040.00",+0.50%,+1.33%,+1.00%,-2.72%,+1.84%,+1.84%
東部ネットワーク (9036),9036.T,"1,237.00",+0.00%,-2.37%,+0.08%,-8.23%,+3.00%,+3.00%
ニトリHD (9843),9843.T,"2,440.00",+1.79%,+2.50%,+3.48%,-6.82%,+10.08%,+10.08%
MRK HD (9980),9980.T,91.00,+0.00%,-2.15%,-3.19%,-9.00%,-7.14%,-7.14%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
iシェアーズ コア米国総合債券ETF (AGG),AGG,99.34,+0.09%,+0.64%,+0.44%,+0.68%,+0.91%,+0.91%


In [3]:
MARKET_TICKERS = {
    "Singapore Healthcare": {
        # --- 病院経営・総合クリニック ---
        "Raffles Medical Group (ラッフルズ医療)": "BSL.SI",  # 【修正】R01.SI から変更
        "IHH Healthcare (マウント・エリザベス等経営)": "Q0F.SI",  # 【修正】I01.SI から変更
        "Thomson Medical Group (トムソン医療)": "A50.SI",
        
        # --- 医療不動産（REIT / 病院・介護施設） ---
        "Parkway Life REIT (パークウェイ・ライフ)": "C2PU.SI",
        "First REIT (ファースト・リート)": "AW9U.SI",
        
        # --- 専門医療・歯科クリニックチェーン ---
        "Q&M Dental Group (歯科大手チェーン)": "QC7.SI",
        "Alliance Healthcare Group (総合診療・クリニック)": "MIJ.SI",  # 【修正】1D8から変更
        "Medinex Limited (医療サポート・クリニック運営)": "OTX.SI",
        
        # --- 医療用消耗品・製薬 ---
        "Medtecs International (医療用消耗品)": "546.SI",
        "Hyphens Pharma (医薬品・皮膚科製品)": "1J5.SI"
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Raffles Medical Group (ラッフルズ医療),BSL.SI,0.92,-1.08%,-0.54%,-1.08%,-2.65%,-4.65%,-4.65%
IHH Healthcare (マウント・エリザベス等経営),Q0F.SI,2.72,-1.09%,+1.49%,-1.45%,-4.56%,-4.23%,-4.23%
Thomson Medical Group (トムソン医療),A50.SI,0.05,+0.00%,+0.00%,-1.85%,-5.36%,-7.02%,-7.02%
Parkway Life REIT (パークウェイ・ライフ),C2PU.SI,4.09,+0.25%,+2.25%,+3.81%,+2.76%,+1.74%,+1.74%
First REIT (ファースト・リート),AW9U.SI,0.22,-2.22%,-2.22%,-4.35%,-6.38%,-6.42%,-6.42%
Q&M Dental Group (歯科大手チェーン),QC7.SI,0.56,-1.75%,-2.61%,-2.61%,-6.67%,-4.27%,-4.27%
Alliance Healthcare Group (総合診療・クリニック),MIJ.SI,0.16,+0.00%,+0.00%,+0.00%,+0.63%,-2.45%,-2.45%
Medinex Limited (医療サポート・クリニック運営),OTX.SI,0.22,+0.00%,+0.00%,+0.00%,-10.00%,-2.17%,-2.17%
Medtecs International (医療用消耗品),546.SI,0.12,+0.86%,+0.86%,-1.68%,-7.14%,-0.85%,-0.85%
Hyphens Pharma (医薬品・皮膚科製品),1J5.SI,0.35,+0.00%,+2.94%,+1.45%,+7.69%,+4.41%,+4.41%


In [4]:
MARKET_TICKERS = {
"Singapore Food & Agribusiness": {
        # --- 国際的アグリビジネス（巨大穀物・商社） ---
        "Wilmar International (ウィルマー・世界最大級のパーム油)": "F34.SI",
        "Olam Group (オラム・ココア、コーヒー、ナッツ等大手)": "VC2.SI",
        "Olam Agri Holdings (オラムのアグリ部門 ※上場状況による)": "OAG.SI",
        
        # --- パーム油・農園経営（プランテーション） ---
        "Golden Agri-Resources (ゴールデン・アグリ・パーム油大手)": "E5H.SI",
        "First Resources (ファースト・リソーシズ・パーム油生産)": "EB5.SI",
        "Bumitama Agri (ブミタマ・アグリ・インドネシア農園)": "P8D.SI",
        
        # --- 食品製造・流通・飲料 ---
        "Fraser and Neave (F&N・飲料・乳製品大手)": "F99.SI",
        "Japfa (ジャプファ・乳肉製品、畜産大手)": "UD2.SI",
        "Delfi Limited (デルフィ・チョコレート製造・流通)": "P34.SI",
        "Sheng Siong Group (シェンシオン・食品スーパー大手)": "OV8.SI"
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Wilmar International (ウィルマー・世界最大級のパーム油),F34.SI,3.66,-1.35%,-1.88%,-1.88%,+7.96%,+1.39%,+1.39%
Olam Group (オラム・ココア、コーヒー、ナッツ等大手),VC2.SI,1.21,-2.42%,-4.72%,-5.47%,+1.68%,+18.63%,+18.63%
Golden Agri-Resources (ゴールデン・アグリ・パーム油大手),E5H.SI,0.27,+1.89%,+0.00%,+0.00%,+0.00%,-11.89%,-11.89%
First Resources (ファースト・リソーシズ・パーム油生産),EB5.SI,3.12,-0.95%,-1.27%,-1.58%,+16.85%,-8.10%,-8.10%
Fraser and Neave (F&N・飲料・乳製品大手),F99.SI,1.42,-0.70%,-0.70%,-1.39%,-0.70%,-1.04%,-1.04%
Delfi Limited (デルフィ・チョコレート製造・流通),P34.SI,0.89,-0.56%,-1.66%,-3.78%,-5.32%,-14.30%,-14.30%
Sheng Siong Group (シェンシオン・食品スーパー大手),OV8.SI,3.22,-0.62%,-0.62%,-1.23%,+6.27%,+6.27%,+6.27%
